Оценка одной RAG-системы через RAGAS.

Один переключатель `LLM_BACKEND` управляет и генерацией в проверяемой RAG-системе, и judge-моделью RAGAS.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from ragas.run_config import RunConfig

from rag_eval.run_eval import latest_xlsx, run_single_rag_eval
from rag_systems.gigachat_bge_m3 import RAG_SYSTEM_NAME, build_rag_system

# ===== Пути и входные данные =====
# DATA_DIR: папка с золотыми вопросами (.xlsx).
DATA_DIR = Path('data')
# REPORTS_DIR: куда сохраняются все артефакты оценки (scores, summary, report.html).
REPORTS_DIR = Path('reports')
# GOLD_XLSX: автоматически выбирается самый свежий .xlsx из DATA_DIR.
GOLD_XLSX = latest_xlsx(DATA_DIR)
# GOLD_LIMIT: ограничить число строк для быстрого прогона (0 или <0 = без ограничения).
GOLD_LIMIT = int(os.environ.get('GOLD_LIMIT', '2'))
if GOLD_LIMIT > 0:
    _gold_df = pd.read_excel(GOLD_XLSX).head(GOLD_LIMIT)
    GOLD_XLSX = DATA_DIR / f'{GOLD_XLSX.stem}_top{GOLD_LIMIT}.xlsx'
    _gold_df.to_excel(GOLD_XLSX, index=False)

# ===== Выбор backend =====
# Единый backend для генерации RAG-ответа и judge-оценки RAGAS: 'koboldcpp' или 'gigachat'.
LLM_BACKEND = os.environ.get('LLM_BACKEND', 'koboldcpp')
# Если 1, принудительно разрешен только локальный backend koboldcpp.
ISOLATED_LOCAL_ONLY = os.environ.get('ISOLATED_LOCAL_ONLY', '1') == '1'
if ISOLATED_LOCAL_ONLY and LLM_BACKEND != 'koboldcpp':
    raise ValueError("ISOLATED_LOCAL_ONLY=1 разрешает только LLM_BACKEND='koboldcpp'")

# ===== Таймауты и параллелизм =====
# Таймаут задачи RAGAS целиком (сек) для одной строки/метрики.
RAGAS_TIMEOUT_SECONDS = float(os.environ.get('RAGAS_TIMEOUT_SECONDS', '1800'))
# Число параллельных воркеров RAGAS (1 = последовательный режим, наиболее стабильный).
RAGAS_MAX_WORKERS = int(os.environ.get('RAGAS_MAX_WORKERS', '1'))
# Таймаут LLM самой RAG-системы (используется в rag_systems/gigachat_bge_m3.py).
os.environ.setdefault('MODEL_TIMEOUT_SECONDS', '0')

# Конфиг выполнения RAGAS: timeout/parallelism.
ragas_run_config = RunConfig(timeout=RAGAS_TIMEOUT_SECONDS, max_workers=RAGAS_MAX_WORKERS)

# ===== Проверяемая RAG-система =====
# build_rag_system берет retrieval/LLM настройки из rag_systems/gigachat_bge_m3.py и env-переменных.
rag_system = build_rag_system(llm_backend=LLM_BACKEND)
print('Backend:', LLM_BACKEND)
print('RAGAS timeout (s):', RAGAS_TIMEOUT_SECONDS)
print('Shared LLM/BGE mode: ON')



In [ ]:
# Запуск оценки одной RAG-системы.
run_dir = run_single_rag_eval(
    # Путь к xlsx с вопросами/эталонами.
    gold_path=GOLD_XLSX,
    # Экземпляр тестируемой RAG-системы (генерация + retrieval).
    rag_system=rag_system,
    # По умолчанию run_single_rag_eval использует shared LLM/BGE из rag_system.
    # RunConfig для таймаутов и параллелизма RAGAS.
    ragas_run_config=ragas_run_config,
    # Корневая папка, куда будет сохранен run.
    reports_dir=REPORTS_DIR,
    # Имя системы в названии папки отчета.
    run_name=RAG_SYSTEM_NAME,
)

print('Saved:', run_dir.resolve())

